In [5]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/opt/spark-3.3.0-bin-hadoop3"

import findspark
findspark.init('/opt/spark-3.3.0-bin-hadoop3')

from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col
from datetime import datetime, timedelta

# Sessão Spark
spark = SparkSession.builder \
    .appName("EcommerceDataLake") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.hadoop_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.hadoop_catalog.type", "hadoop") \
    .config("spark.sql.catalog.hadoop_catalog.warehouse", "/home/tavares/warehouse") \
    .config("spark.sql.default.catalog", "hadoop_catalog") \
    .getOrCreate()

In [7]:
spark.sql("""
CREATE TABLE hadoop_catalog.default.vendas_ecommerce (
    venda_id INT,
    produto_nome STRING,
    categoria STRING,
    quantidade INT,
    preco_unitario DOUBLE,
    data_venda DATE,
    cliente_id STRING,
    vendedor_id INT
)
USING iceberg
""")

DataFrame[]

In [9]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(1, 'Notebook Dell', 'Eletrônicos', 2, 2500.00, CAST('2023-01-15' AS DATE), 'CLI001', 101),
(2, 'Mouse Logitech', 'Eletrônicos', 5, 80.00, CAST('2023-01-16' AS DATE), 'CLI002', 102),
(3, 'Mesa Escritório', 'Móveis', 1, 800.00, CAST('2023-02-10' AS DATE), 'CLI003', 101),
(4, 'Cadeira Gamer', 'Móveis', 2, 600.00, CAST('2023-02-15' AS DATE), 'CLI001', 103),
(5, 'Smartphone Samsung', 'Eletrônicos', 1, 1200.00, CAST('2023-03-20' AS DATE), 'CLI004', 102)
""")

DataFrame[]

In [11]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(6, 'Tablet iPad', 'Eletrônicos', 1, 3000.00, CAST('2024-01-10' AS DATE), 'CLI002', 101),
(7, 'Sofá 3 Lugares', 'Móveis', 1, 1500.00, CAST('2024-01-20' AS DATE), 'CLI005', 103),
(8, 'Monitor 4K', 'Eletrônicos', 2, 800.00, CAST('2024-02-05' AS DATE), 'CLI003', 102)
""")

DataFrame[]

In [12]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce.history
""").show(truncate=False)

+-----------------------+-------------------+------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id         |is_current_ancestor|
+-----------------------+-------------------+------------------+-------------------+
|2026-03-24 23:52:30.619|275613150865179132 |null              |true               |
|2026-03-24 23:56:31.765|2315462978858892894|275613150865179132|true               |
+-----------------------+-------------------+------------------+-------------------+



In [13]:
df_snapshots = spark.sql("""
SELECT snapshot_id 
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
ORDER BY committed_at
""")

primeiro_snapshot = df_snapshots.first()["snapshot_id"]

print(primeiro_snapshot)

275613150865179132


In [14]:
spark.sql(f"""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
VERSION AS OF {primeiro_snapshot}
""").show()

+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+



In [15]:
spark.sql("""
SELECT * 
FROM hadoop_catalog.default.vendas_ecommerce
""").show()

+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|venda_id|      produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|
+--------+------------------+-----------+----------+--------------+----------+----------+-----------+
|       1|     Notebook Dell|Eletrônicos|         2|        2500.0|2023-01-15|    CLI001|        101|
|       2|    Mouse Logitech|Eletrônicos|         5|          80.0|2023-01-16|    CLI002|        102|
|       3|   Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|
|       4|     Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|
|       5|Smartphone Samsung|Eletrônicos|         1|        1200.0|2023-03-20|    CLI004|        102|
|       6|       Tablet iPad|Eletrônicos|         1|        3000.0|2024-01-10|    CLI002|        101|
|       7|    Sofá 3 Lugares|     Móveis|         1|        1500.0|2024-01-20|    

In [16]:
# Atual
spark.sql("""
SELECT COUNT(*) AS total_atual 
FROM hadoop_catalog.default.vendas_ecommerce
""").show()

# Snapshot antigo
spark.sql(f"""
SELECT COUNT(*) AS total_2023
FROM hadoop_catalog.default.vendas_ecommerce
VERSION AS OF {primeiro_snapshot}
""").show()

+-----------+
|total_atual|
+-----------+
|          8|
+-----------+

+----------+
|total_2023|
+----------+
|         5|
+----------+



In [17]:
spark.sql("""
ALTER TABLE hadoop_catalog.default.vendas_ecommerce
ADD COLUMNS (
    desconto DOUBLE,
    canal_venda STRING
)
""")

DataFrame[]

In [18]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, CAST('2024-03-15' AS DATE), 'CLI006', 101, 10.0, 'online'),
(10, 'Mesa Centro', 'Móveis', 1, 400.00, CAST('2024-03-20' AS DATE), 'CLI007', 102, 5.0, 'loja_fisica')
""")

DataFrame[]

In [22]:
spark.sql("""
INSERT INTO hadoop_catalog.default.vendas_ecommerce VALUES
(9, 'Headset Gamer', 'Eletrônicos', 3, 250.00, CAST('2024-03-15' AS DATE), 'CLI006', 101, 10.0, 'online'),
(10, 'Mesa Centro', 'Móveis', 1, 400.00, CAST('2024-03-20' AS DATE), 'CLI007', 102, 5.0, 'loja_fisica')
""")


DataFrame[]

In [23]:
spark.sql("""
SELECT 
    venda_id,
    data_venda,
    desconto,
    canal_venda
FROM hadoop_catalog.default.vendas_ecommerce
WHERE venda_id <= 8
ORDER BY venda_id
""").show()

+--------+----------+--------+-----------+
|venda_id|data_venda|desconto|canal_venda|
+--------+----------+--------+-----------+
|       1|2023-01-15|    null|       null|
|       2|2023-01-16|    null|       null|
|       3|2023-02-10|    null|       null|
|       4|2023-02-15|    null|       null|
|       5|2023-03-20|    null|       null|
|       6|2024-01-10|    null|       null|
|       7|2024-01-20|    null|       null|
|       8|2024-02-05|    null|       null|
+--------+----------+--------+-----------+



In [24]:
spark.sql("""
UPDATE hadoop_catalog.default.vendas_ecommerce
SET preco_unitario = preco_unitario * 100
WHERE categoria = 'Eletrônicos'
""")

DataFrame[]

In [25]:
spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation
FROM hadoop_catalog.default.vendas_ecommerce.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|5896822337524993285|2026-03-25 00:10:02.271|overwrite|
|8248218802939759359|2026-03-25 00:07:28.385|append   |
|663467306352342892 |2026-03-25 00:01:59.838|append   |
|2315462978858892894|2026-03-24 23:56:31.765|append   |
|275613150865179132 |2026-03-24 23:52:30.619|append   |
+-------------------+-----------------------+---------+



In [26]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW vendas_updates AS
SELECT 1 as venda_id, 'Notebook Dell UPDATED' as produto_nome, 'Eletrônicos' as categoria, 
       2 as quantidade, 2600.00 as preco_unitario, CAST('2023-01-15' AS DATE) as data_venda, 
       'CLI001' as cliente_id, 101 as vendedor_id, 0.0 as desconto, 'online' as canal_venda
UNION ALL
SELECT 11 as venda_id, 'Teclado Mecânico' as produto_nome, 'Eletrônicos' as categoria,
       4 as quantidade, 300.00 as preco_unitario, CAST('2024-04-01' AS DATE) as data_venda,
       'CLI008' as cliente_id, 103 as vendedor_id, 15.0 as desconto, 'online' as canal_venda
""")

DataFrame[]

In [27]:
spark.sql("""
MERGE INTO hadoop_catalog.default.vendas_ecommerce AS target
USING vendas_updates AS source
ON target.venda_id = source.venda_id

WHEN MATCHED THEN UPDATE SET
    target.produto_nome = source.produto_nome,
    target.categoria = source.categoria,
    target.quantidade = source.quantidade,
    target.preco_unitario = source.preco_unitario,
    target.data_venda = source.data_venda,
    target.cliente_id = source.cliente_id,
    target.vendedor_id = source.vendedor_id,
    target.desconto = source.desconto,
    target.canal_venda = source.canal_venda

WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[]

In [28]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
WHERE venda_id IN (1, 11)
""").show()

+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|venda_id|        produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|desconto|canal_venda|
+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|       1|Notebook Dell UPD...|Eletrônicos|         2|        2600.0|2023-01-15|    CLI001|        101|     0.0|     online|
|      11|    Teclado Mecânico|Eletrônicos|         4|         300.0|2024-04-01|    CLI008|        103|    15.0|     online|
+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+



In [30]:
spark.sql("""
SELECT COUNT(*) AS total_arquivos
FROM hadoop_catalog.default.vendas_ecommerce.files
""").show()

+--------------+
|total_arquivos|
+--------------+
|             8|
+--------------+



In [31]:
spark.sql("""
SELECT 
    COUNT(*) AS total_arquivos,
    ROUND(AVG(file_size_in_bytes)/1024,2) AS tamanho_medio_kb,
    ROUND(MAX(file_size_in_bytes)/1024,2) AS maior_arquivo_kb,
    ROUND(MIN(file_size_in_bytes)/1024,2) AS menor_arquivo_kb
FROM hadoop_catalog.default.vendas_ecommerce.files
""").show()

+--------------+----------------+----------------+----------------+
|total_arquivos|tamanho_medio_kb|maior_arquivo_kb|menor_arquivo_kb|
+--------------+----------------+----------------+----------------+
|             8|            2.77|            2.95|            2.66|
+--------------+----------------+----------------+----------------+



In [34]:
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes,
    record_count
FROM hadoop_catalog.default.vendas_ecommerce.files
""").show(truncate=False)

+--------------------------------------------------------------------------------------------------------------------+------------------+------------+
|file_path                                                                                                           |file_size_in_bytes|record_count|
+--------------------------------------------------------------------------------------------------------------------+------------------+------------+
|/home/tavares/warehouse/default/vendas_ecommerce/data/00000-229-514ca865-47ac-415f-8abd-f759191ea436-0-00001.parquet|3550              |13          |
+--------------------------------------------------------------------------------------------------------------------+------------------+------------+



In [33]:
spark.sql("""
CALL hadoop_catalog.system.rewrite_data_files(
    table => 'default.vendas_ecommerce'
)
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint]

In [35]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce.partitions
""").show(truncate=False)

+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|13          |1         |3550                         |0                           |0                         |0                           |0                         |2026-03-25 00:26:52.568|5246724965297846523     |
+------------+----------+-----------------------------+----------------------------+--------------------------+---------------------

In [38]:
antes = spark.sql("""
SELECT COUNT(*) AS total_arquivos
FROM hadoop_catalog.default.vendas_ecommerce.files
""")

antes.show()

+--------------+
|total_arquivos|
+--------------+
|             1|
+--------------+



In [39]:
spark.sql("""
CALL hadoop_catalog.system.rewrite_data_files(
    table => 'default.vendas_ecommerce'
)
""")

DataFrame[rewritten_data_files_count: int, added_data_files_count: int, rewritten_bytes_count: bigint]

In [40]:
depois = spark.sql("""
SELECT COUNT(*) AS total_arquivos
FROM hadoop_catalog.default.vendas_ecommerce.files
""")

depois.show()

+--------------+
|total_arquivos|
+--------------+
|             1|
+--------------+



In [41]:
spark.sql("""
SELECT *
FROM hadoop_catalog.default.vendas_ecommerce
ORDER BY venda_id
""").show()

+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|venda_id|        produto_nome|  categoria|quantidade|preco_unitario|data_venda|cliente_id|vendedor_id|desconto|canal_venda|
+--------+--------------------+-----------+----------+--------------+----------+----------+-----------+--------+-----------+
|       1|Notebook Dell UPD...|Eletrônicos|         2|        2600.0|2023-01-15|    CLI001|        101|     0.0|     online|
|       2|      Mouse Logitech|Eletrônicos|         5|        8000.0|2023-01-16|    CLI002|        102|    null|       null|
|       3|     Mesa Escritório|     Móveis|         1|         800.0|2023-02-10|    CLI003|        101|    null|       null|
|       4|       Cadeira Gamer|     Móveis|         2|         600.0|2023-02-15|    CLI001|        103|    null|       null|
|       5|  Smartphone Samsung|Eletrônicos|         1|      120000.0|2023-03-20|    CLI004|        102|    null|       null|
